The program contains a ML model trained on 5 years of Nvidia price data.
The model makes guesses using the 10 day simple moving average and the price close of the day

The hypothesis is that as a 10 day moving average is highly correlated to the price the model will likely have a decent accuracy.
This does not mean the model would in action make money.

In [ ]:
import torch
import yfinance as yf

In [ ]:
#Loading the appropriate training data for the model and also constructing a 10 day moving average

NVDA_train = yf.Ticker('NVDA').history(start = "2015-01-01",end = "2020-01-01", interval = "1d")[["Close"]]
NVDA_train["SMA_10"] = NVDA_train["Close"].rolling(10).mean()
NVDA_train["Next_close"] = NVDA_train['Close'].shift(-1)

NVDA_train = NVDA_train.dropna()

X = torch.tensor(NVDA_train[["Close", "SMA_10"]].values, dtype=torch.float32)
y_true = torch.tensor(NVDA_train["Next_close"].values,dtype=torch.float32).unsqueeze(1)



In [45]:
#Model

import torch.nn as nn

class LinearRegressionModel(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear_layer = nn.Linear(in_features, out_features)
    
    def forward(self,x):
        return self.linear_layer(x)
    
model =LinearRegressionModel(in_features=2, out_features=1)

print(model)

LinearRegressionModel(
  (linear_layer): Linear(in_features=2, out_features=1, bias=True)
)


In [46]:
#Weight updater

import torch.optim as optim

learning_rate = 0.001

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

#loss calclulator

loss_fn = nn.MSELoss()


In [47]:
#Training loop 

epochs = 10000

for epoch in range(epochs):
    y_hat = model(X)

    loss = loss_fn(y_hat, y_true)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0 or epoch == 0:
        print(f"Epoch: {epoch:02d}: Loss = {loss.item():.4f}")
    


Epoch: 00: Loss = 47.9171
Epoch: 100: Loss = 37.2978
Epoch: 200: Loss = 28.5471
Epoch: 300: Loss = 21.4595
Epoch: 400: Loss = 15.8166
Epoch: 500: Loss = 11.4096
Epoch: 600: Loss = 8.0407
Epoch: 700: Loss = 5.5261
Epoch: 800: Loss = 3.6977
Epoch: 900: Loss = 2.4059
Epoch: 1000: Loss = 1.5213
Epoch: 1100: Loss = 0.9356
Epoch: 1200: Loss = 0.5616
Epoch: 1300: Loss = 0.3317
Epoch: 1400: Loss = 0.1959
Epoch: 1500: Loss = 0.1190
Epoch: 1600: Loss = 0.0772
Epoch: 1700: Loss = 0.0553
Epoch: 1800: Loss = 0.0443
Epoch: 1900: Loss = 0.0389
Epoch: 2000: Loss = 0.0361
Epoch: 2100: Loss = 0.0346
Epoch: 2200: Loss = 0.0336
Epoch: 2300: Loss = 0.0329
Epoch: 2400: Loss = 0.0322
Epoch: 2500: Loss = 0.0315
Epoch: 2600: Loss = 0.0308
Epoch: 2700: Loss = 0.0301
Epoch: 2800: Loss = 0.0295
Epoch: 2900: Loss = 0.0288
Epoch: 3000: Loss = 0.0281
Epoch: 3100: Loss = 0.0274
Epoch: 3200: Loss = 0.0268
Epoch: 3300: Loss = 0.0261
Epoch: 3400: Loss = 0.0255
Epoch: 3500: Loss = 0.0248
Epoch: 3600: Loss = 0.0242
Epoch:

In [60]:
#Evaluation data

NVDA_ev = yf.Ticker('NVDA').history(start = "2020-01-01",end = "2025-01-01", interval = "1d")[["Close"]]
NVDA_ev["SMA_10"] = NVDA_ev["Close"].rolling(10).mean()

NVDA_ev = NVDA_ev.dropna()

X_ev = torch.tensor(NVDA_ev[["Close", "SMA_10"]].values, dtype=torch.float32)
y = torch.tensor(NVDA_ev["Close"].shift(-1).values, dtype=torch.float32)

In [ ]:
#Running the model on evaluation data and storing it with the actual results

output = {}

with torch.no_grad():
    output["Prediction"] = model(X_ev)
    output["Actual"] = y



In [81]:
#Cleaning up and making the data usable
output["Prediction"] = model(X_ev).detach().numpy().flatten()
output["Actual"] = y.detach().numpy()

output["Actual"] = output["Actual"][:-1]
output["Prediction"] = output["Prediction"][:-1]


In [85]:
#To put a number on the accuracy of the model I'll be using RMSE, it tells how many dollars off the prediction was on average
import numpy as np
RMSE = np.sqrt(np.mean((output["Actual"]-output["Prediction"])**2))

Avrg_price = np.mean(NVDA_ev["Close"])

Acc = (1 - RMSE/Avrg_price) * 100

print(RMSE)
print(Avrg_price)
print(Acc)

2.0296714
38.62184561644677
94.74475805546835


So, with the RMSE being 2.0296714, it means that during the priceaction of 2020-2025 the models predictions were off by ~2.03$.
During the same period the averaged out price for Nvidia stock was 38.62184561644677.

This would mean the model predicted the price with an on average 94.74% accuracy.

Of course it is worth noting that the model only considers the 10 sma that is high correlation to the price.

In conclusion the model works but likely in a real world setting won't grant outsized returns